In [ ]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from tqdm import tqdm

train_images = np.load("./data_set/train_images.npy")
train_labels = np.load("./data_set/train_labels.npy")
test_images = np.load("./data_set/test_images.npy")
test_labels = np.load("./data_set/test_labels.npy")

# Încărcăm dicționarul/lista de patch-uri vizuale (echivalentul words.txt / n-gramelor)
patch_filters = np.load("./data_set/patch_filters.npy")




In [ ]:
def extract_2d_patch_frequencies(image, filters, threshold=0.9):
    """
    Extrage toate ferestrele PxP dintr-o imagine și calculează
    frecvența apariției fiecărui filtru (patch de referință).
    """
    K, P, _ = filters.shape
    H, W = image.shape

    # Aplatizăm filtrele pentru a putea face înmulțire matriceală rapidă: (K, P*P)
    filters_flat = filters.reshape(K, -1)

    # Extragem toate ferestrele posibile de dimensiune PxP din imagine
    windows = []
    for i in range(H - P + 1):
        for j in range(W - P + 1):
            window = image[i:i+P, j:j+P].flatten()
            windows.append(window)
    windows_flat = np.array(windows) # formă: (Număr_Ferestre, P*P)

    # Calculăm similaritatea cosinus (produsul scalar normalizat)
    # 1. Normalizăm ferestrele și filtrele
    norm_windows = np.linalg.norm(windows_flat, axis=1, keepdims=True)
    norm_windows[norm_windows == 0] = 1e-10 # Evităm împărțirea la zero

    norm_filters = np.linalg.norm(filters_flat, axis=1, keepdims=True)
    norm_filters[norm_filters == 0] = 1e-10

    windows_normalized = windows_flat / norm_windows
    filters_normalized = filters_flat / norm_filters

    # 2. Produs scalar între toate ferestrele și toate filtrele
    # similaritati va avea forma (Număr_Ferestre, K)
    similarities = np.dot(windows_normalized, filters_normalized.T)

    # 3. Numărăm de câte ori similaritatea depășește pragul stabilit
    frequencies = np.sum(similarities > threshold, axis=0)

    return frequencies

def apply_patch_extraction(images, filters, threshold=0.9):
    features = []
    print("Extragere patch frequencies...")
    for img in tqdm(images):
        freq = extract_2d_patch_frequencies(img, filters, threshold)
        features.append(freq)
    return np.array(features)

# Generăm matricile de antrenare și testare bazate pe frecvențe
train_data = apply_patch_extraction(train_images, patch_filters, threshold=0.9)
test_data = apply_patch_extraction(test_images, patch_filters, threshold=0.9)

In [ ]:
# ==========================================
# 3. Modelul KNN (K-Nearest Neighbors)
# ==========================================
print("\n--- KNN ---")
knn = KNeighborsClassifier(n_neighbors=5, metric='manhattan')
knn.fit(train_data, train_labels)
knn_preds = knn.predict(test_data)
print(f"Accuracy KNN: {np.mean(knn_preds == test_labels):.4f}")

In [ ]:
# ==========================================
# 4. Modelul KRR (Kernel Ridge Regression)
# ==========================================
print("\n--- KRR ---")
# Presupunem 4 clase (0, 1, 2, 3) - adaptat din logica One-vs-All din 2025
num_classes = 4
predictions_krr = []

for i in range(num_classes):
    # Transformăm etichetele în +1 pentru clasa curentă și -1 pentru restul
    new_train_labels = ((train_labels == i) * 2) - 1

    krr = KernelRidge(kernel="linear", alpha=100)
    krr.fit(train_data, new_train_labels)

    # Păstrăm scorurile continue prezise pentru test
    predictions_krr.append(krr.predict(test_data))

predictions_krr = np.array(predictions_krr)
# Clasa finală este cea cu scorul maxim pe coloană
final_krr_preds = np.argmax(predictions_krr, axis=0)
print(f"Accuracy KRR: {np.mean(final_krr_preds == test_labels):.4f}")

In [ ]:
# ==========================================
# 5. Modelul SVM cu Kernel Precalculat (Hellinger)
# ==========================================
print("\n--- SVM Precomputed ---")

def hellinger_kernel(features1, features2):
    # Kernelul Hellinger măsoară similaritatea între distribuții de probabilitate/frecvențe
    # Se aplică radical pe elemente și apoi se face produsul scalar
    feat1_sqrt = np.sqrt(features1)
    feat2_sqrt = np.sqrt(features2)
    matrix = np.matmul(feat1_sqrt, feat2_sqrt.T)
    return matrix

# Calculăm matricile kernel precomputed
train_matrix_kernel = hellinger_kernel(train_data, train_data)
test_matrix_kernel = hellinger_kernel(test_data, train_data)

svm_model = SVC(kernel='precomputed', C=10)
svm_model.fit(train_matrix_kernel, train_labels)
svm_preds = svm_model.predict(test_matrix_kernel)

print(f"Accuracy SVM: {np.mean(svm_preds == test_labels):.4f}")

In [ ]:
# IN CAZ DE PICA POZE

import os
import cv2
import numpy as np

def incarca_date_brute(cale_folder_imagini, cale_fisier_index):
    imagini = []
    etichete = []

    # Deschidem fișierul care face legătura între poză și clasă (ex: train.txt)
    with open(cale_fisier_index, 'r') as f:
        # Trecem peste prima linie dacă are header (ex: "nume_fisier,eticheta")
        linii = f.readlines()[1:]

    for linie in linii:
        # Despărțim linia pe baza virgulei
        elemente = linie.strip().split(',')
        nume_fisier = elemente[0]
        eticheta = int(elemente[1])

        # Citim imaginea în format alb-negru (Grayscale) pentru a o menține 2D
        cale_completa = os.path.join(cale_folder_imagini, nume_fisier)
        img = cv2.imread(cale_completa, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            # Opțional: putem da un resize aici dacă pozele au dimensiuni diferite
            # img = cv2.resize(img, (64, 64))
            imagini.append(img)
            etichete.append(eticheta)

    # Returnăm tensorii pregătiți pentru antrenare
    return np.array(imagini), np.array(etichete)

# Utilizare:
# train_images, train_labels = incarca_date_brute('./data/train/', './data/train.txt')